# Capstone Assignment: Credit Card Fraud Detection
### CS2227: Artificial Intelligence and Machine Learning

---

## Problem Statement

Real-world credit card datasets are **massively imbalanced** — typically only ~0.17% of transactions
are fraudulent. This means a naive model that predicts "Legitimate" for every transaction would
achieve 99.83% accuracy — yet catch **zero fraud cases**. This is the **Accuracy Paradox**.

Our pipeline synthesizes both labs:
- **Lab 4 (SMOTE):** Correct the class imbalance in the training set
- **Lab 5 (SVM + RBF Kernel):** Handle the non-linear decision boundary between fraud and legitimate transactions

## Pipeline Overview

```
Raw Dataset
    │
    ├─── EDA (prove imbalance)
    │
    ├─── Train / Test Split  ← SMOTE applied AFTER this step only
    │         │
    │    X_train, y_train ──► SMOTE ──► X_train_resampled
    │    X_test,  y_test  ──► (untouched, real-world distribution preserved)
    │
    ├─── StandardScaler (fit on train, transform both)
    │
    ├─── Model 1: Logistic Regression
    ├─── Model 2: SVM with RBF Kernel
    │
    └─── Evaluation: Precision, Recall, F1-Score (NOT Accuracy)
```

---

## Why NOT Accuracy?

With 99.83% legitimate transactions, a model predicting all-legitimate gets **99.83% accuracy**
but **0% Recall on fraud** — it catches nothing. We use:

- **Precision** = Of all transactions flagged as fraud, how many actually were?
  → Minimizes false alarms (customer inconvenience)
- **Recall** = Of all actual frauds, how many did we catch?
  → Minimizes missed fraud (financial loss)
- **F1-Score** = Harmonic mean of Precision and Recall
  → Balanced view when classes are imbalanced

> 🏦 **For a bank, Recall is more important than Precision.**
> Missing a fraud case (False Negative) costs the bank real money and destroys customer trust.
> A false alarm (False Positive) is inconvenient but recoverable — the customer can confirm the transaction.

---
## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install imbalanced-learn -q
!pip install kagglehub -q

In [ ]:
# --- Standard Library Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# --- Scikit-learn Imports ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

# --- Imbalanced-learn Imports ---
from imblearn.over_sampling import SMOTE

print("All libraries imported successfully.")

---
## Step 2: Load the Dataset

We use the **Credit Card Fraud Detection** dataset from Kaggle (by ULB Machine Learning Group).
It contains 284,807 transactions made by European cardholders in September 2013.
Only 492 (0.172%) are fraudulent.

Features V1–V28 are PCA-transformed (anonymised for privacy). Only `Time` and `Amount` are raw.
`Class` is the target: 0 = Legitimate, 1 = Fraud.

In [ ]:
# ---------------------------------------------------------------
# Download the dataset using kagglehub
# If this fails in your environment, manually upload creditcard.csv
# from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
# ---------------------------------------------------------------
import kagglehub

try:
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    df = pd.read_csv(f"{path}/creditcard.csv")
    print(f"Dataset loaded via kagglehub from: {path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    print("Trying fallback: upload creditcard.csv manually and run:")
    print("  df = pd.read_csv('creditcard.csv')")
    # --- FALLBACK: uncomment the line below and upload the file manually ---
    # df = pd.read_csv('creditcard.csv')

# Quick preview of the dataset
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

---
## Step 3: Exploratory Data Analysis (EDA)

Before modelling, we must **understand and visualize** the data — especially the class imbalance.

In [ ]:
# ---------------------------------------------------------------
# 3a. Basic Dataset Information
# ---------------------------------------------------------------
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total transactions : {len(df):,}")
print(f"Total features     : {df.shape[1]}")
print(f"Missing values     : {df.isnull().sum().sum()}")
print(f"Data types         : {df.dtypes.value_counts().to_dict()}")

print("\n--- Class Distribution ---")
class_counts = df['Class'].value_counts()
print(f"  Legitimate (0) : {class_counts[0]:,} ({class_counts[0]/len(df)*100:.4f}%)")
print(f"  Fraud      (1) : {class_counts[1]:,} ({class_counts[1]/len(df)*100:.4f}%)")
print(f"  Imbalance Ratio: {class_counts[0]/class_counts[1]:.0f}:1  (legitimate:fraud)")

In [ ]:
# ---------------------------------------------------------------
# 3b. Visualize Class Imbalance
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploratory Data Analysis: Credit Card Fraud Dataset', fontsize=15, fontweight='bold')

# --- Plot 1: Bar Chart of Class Counts ---
colors = ['#4C72B0', '#DD8452']
bars = axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution (Raw Counts)', fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{count:,}', ha='center', fontsize=10)

# --- Plot 2: Pie Chart showing % split ---
axes[1].pie(
    class_counts.values,
    labels=['Legitimate', 'Fraud'],
    autopct='%1.3f%%',
    colors=colors,
    startangle=90,
    explode=(0, 0.1)    # Explode fraud slice to highlight it
)
axes[1].set_title('Class Distribution (Percentage)', fontweight='bold')

# --- Plot 3: Transaction Amount Distribution by Class ---
df[df['Class'] == 0]['Amount'].hist(ax=axes[2], bins=50, alpha=0.7, color='#4C72B0', label='Legitimate')
df[df['Class'] == 1]['Amount'].hist(ax=axes[2], bins=50, alpha=0.7, color='#DD8452', label='Fraud')
axes[2].set_title('Transaction Amount by Class', fontweight='bold')
axes[2].set_xlabel('Transaction Amount ($)')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].set_xlim(0, 2500)

plt.tight_layout()
plt.show()

print("Observation: The dataset is severely imbalanced — fraud accounts for only 0.172% of transactions.")
print("A naive 'predict-all-legitimate' model would achieve 99.83% accuracy but catch ZERO frauds.")

In [ ]:
# ---------------------------------------------------------------
# 3c. Statistical Summary: Fraud vs Legitimate
# ---------------------------------------------------------------
# Compare mean values of a few key features across classes
# to identify features with the most discriminative power
# ---------------------------------------------------------------
print("--- Mean Feature Values: Fraud vs Legitimate ---")
comparison = df.groupby('Class')[['Amount', 'V1', 'V2', 'V3', 'V4', 'V14', 'V17']].mean()
comparison.index = ['Legitimate', 'Fraud']
print(comparison.round(3))

print("\nObservation: Features like V14 and V17 show large differences between classes,")
print("suggesting non-linear boundaries that SVM with RBF kernel is well-suited to capture.")

---
## Step 4: Train/Test Split

We split **before** applying SMOTE. This is the most critical data hygiene rule:

> If SMOTE is applied before the split, synthetic points derived from test-set neighbors can enter
> the training set. This causes **data leakage** — the model has indirect knowledge of the test set,
> producing artificially inflated metrics that won't hold in production.

In [ ]:
# ---------------------------------------------------------------
# Prepare Features and Target
# ---------------------------------------------------------------
# Drop the 'Time' column — it's a transaction counter with no predictive value
# Keep 'Amount' and all V1–V28 PCA components
# ---------------------------------------------------------------
X = df.drop(columns=['Class', 'Time'])   # Drop target and non-informative time counter
y = df['Class']                           # Target: 0 = Legitimate, 1 = Fraud

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")

# ---------------------------------------------------------------
# Stratified Train/Test Split
# ---------------------------------------------------------------
# stratify=y ensures the fraud/legitimate ratio is preserved in
# both train and test splits (important for imbalanced data)
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Preserve class ratio in both splits
)

print(f"\nTraining set  : {X_train.shape[0]:,} samples")
print(f"  Fraud      : {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"Test set      : {X_test.shape[0]:,} samples")
print(f"  Fraud      : {y_test.sum():,} ({y_test.mean()*100:.3f}%)")
print("\n✓ Split done. SMOTE will only be applied to X_train/y_train next.")

---
## Step 5: Apply SMOTE to Training Set Only

SMOTE is applied **exclusively to `X_train` and `y_train`**. The test set remains completely untouched,
preserving the real-world class distribution for honest evaluation.

In [ ]:
# ---------------------------------------------------------------
# SMOTE: Oversample the minority (Fraud) class in training data
# ---------------------------------------------------------------
# sampling_strategy='minority': bring minority class count up to
# match the majority class count
# k_neighbors=5: each synthetic point is interpolated using 5
# nearest fraud neighbors
# ---------------------------------------------------------------
smote = SMOTE(
    sampling_strategy='minority',
    random_state=42,
    k_neighbors=5
)

print("Applying SMOTE to training set...")
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE → {Counter(y_train)}")
print(f"After  SMOTE → {Counter(y_train_smote)}")
print(f"\nSynthetic fraud samples created: {Counter(y_train_smote)[1] - Counter(y_train)[1]:,}")
print("\n✓ Test set is untouched — no data leakage.")
print(f"  Test set fraud count: {y_test.sum()} ({y_test.mean()*100:.3f}%) — real-world distribution preserved.")

---
## Step 6: Feature Scaling with StandardScaler

StandardScaler is **mandatory** before SVM. SVM uses Euclidean distances $\|x - x'\|^2$;
unscaled features with larger ranges (like `Amount` in dollars) would dominate the distance
calculation, effectively making other features invisible.

We also scale for Logistic Regression — it helps gradient descent converge faster and avoids
numerical instability with regularization.

In [ ]:
# ---------------------------------------------------------------
# StandardScaler: z = (x - mean) / std
# ---------------------------------------------------------------
# IMPORTANT: fit_transform on training data ONLY.
# Then use .transform() (not fit_transform) on the test set.
# Fitting on test data would be another form of data leakage.
# ---------------------------------------------------------------
scaler = StandardScaler()

# Fit on SMOTE-augmented training data, transform both sets
X_train_scaled = scaler.fit_transform(X_train_smote)   # Fit + transform training
X_test_scaled  = scaler.transform(X_test)               # Transform test ONLY (no fit)

print("Scaling applied.")
print(f"Training set mean (should be ~0): {X_train_scaled.mean(axis=0)[:3].round(4)}...")
print(f"Training set std  (should be ~1): {X_train_scaled.std(axis=0)[:3].round(4)}...")
print(f"\nFinal shapes:")
print(f"  X_train_scaled : {X_train_scaled.shape}")
print(f"  X_test_scaled  : {X_test_scaled.shape}")

---
## Step 7: Train the Models

We train two classifiers:
1. **Logistic Regression** — a linear probabilistic classifier (baseline)
2. **SVM with RBF Kernel** — a powerful non-linear classifier

Both are trained on the SMOTE-balanced, scaled training data.

In [ ]:
# ---------------------------------------------------------------
# Model 1: Logistic Regression (Baseline)
# ---------------------------------------------------------------
# Logistic Regression fits a linear decision boundary:
# P(y=1 | x) = sigmoid(w^T x + b)
#
# max_iter=1000: increase from default 100 to ensure convergence
#   on this high-dimensional dataset
# class_weight='balanced': additionally upweights fraud even after
#   SMOTE, providing extra sensitivity to the minority class
# ---------------------------------------------------------------
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
lr_model.fit(X_train_scaled, y_train_smote)
print("✓ Logistic Regression trained.")

In [ ]:
# ---------------------------------------------------------------
# Model 2: SVM with RBF Kernel
# ---------------------------------------------------------------
# kernel='rbf': maps data into infinite-dimensional Gaussian space
#   K(x, x') = exp(-gamma * ||x - x'||^2)
#   Allows the model to capture highly non-linear boundaries.
#
# C=10: regularization parameter
#   Higher C = smaller margin, fewer misclassifications on training
#   We increase C slightly (from default 1.0) because fraud detection
#   rewards tighter fit to fraud examples.
#
# gamma='scale': gamma = 1 / (n_features * Var(X))
#   Automatically adapts to the number of features and their variance.
#
# class_weight='balanced': further compensates for any residual
#   imbalance, upweighting fraud misclassification in the SVM loss.
# ---------------------------------------------------------------
print("Training SVM with RBF Kernel... (this may take a few minutes on the full dataset)")
svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    class_weight='balanced',
    random_state=42
)
svm_model.fit(X_train_scaled, y_train_smote)
print("✓ SVM (RBF) trained.")

---
## Step 8: Evaluation — Precision, Recall, and F1-Score

> ⚠️ **Accuracy is forbidden here.** Due to the accuracy paradox, a model predicting all transactions
> as legitimate would score 99.83% accuracy while catching 0% of fraud.
>
> We evaluate using `classification_report` which provides:
> - **Precision** = TP / (TP + FP) — of flagged frauds, how many were real?
> - **Recall** = TP / (TP + FN) — of real frauds, how many did we catch?
> - **F1-Score** = 2 × (P × R) / (P + R) — harmonic balance of both

In [ ]:
# ---------------------------------------------------------------
# Evaluate Logistic Regression
# ---------------------------------------------------------------
lr_preds = lr_model.predict(X_test_scaled)

print("=" * 55)
print("MODEL 1: LOGISTIC REGRESSION — Classification Report")
print("=" * 55)
print(classification_report(
    y_test, lr_preds,
    target_names=['Legitimate (0)', 'Fraud (1)']
))

# Extract key fraud-class metrics
lr_precision = precision_score(y_test, lr_preds)
lr_recall    = recall_score(y_test, lr_preds)
lr_f1        = f1_score(y_test, lr_preds)
print(f"Fraud Class → Precision: {lr_precision:.4f} | Recall: {lr_recall:.4f} | F1: {lr_f1:.4f}")

In [ ]:
# ---------------------------------------------------------------
# Evaluate SVM (RBF Kernel)
# ---------------------------------------------------------------
svm_preds = svm_model.predict(X_test_scaled)

print("=" * 55)
print("MODEL 2: SVM (RBF KERNEL) — Classification Report")
print("=" * 55)
print(classification_report(
    y_test, svm_preds,
    target_names=['Legitimate (0)', 'Fraud (1)']
))

svm_precision = precision_score(y_test, svm_preds)
svm_recall    = recall_score(y_test, svm_preds)
svm_f1        = f1_score(y_test, svm_preds)
print(f"Fraud Class → Precision: {svm_precision:.4f} | Recall: {svm_recall:.4f} | F1: {svm_f1:.4f}")

In [ ]:
# ---------------------------------------------------------------
# Confusion Matrix Comparison
# ---------------------------------------------------------------
# True Negatives  (TN): Legitimate predicted as Legitimate  ✓
# False Positives (FP): Legitimate predicted as Fraud       ✗ (False alarm)
# False Negatives (FN): Fraud predicted as Legitimate       ✗ (Missed fraud — most costly!)
# True Positives  (TP): Fraud predicted as Fraud            ✓
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices: Logistic Regression vs SVM (RBF)', fontsize=14, fontweight='bold')

for ax, preds, title in zip(
    axes,
    [lr_preds, svm_preds],
    ['Logistic Regression', 'SVM (RBF Kernel)']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')
    # Annotate FN position
    ax.text(0.5, -0.15, f"False Negatives (missed fraud) = {cm[1][0]}",
            transform=ax.transAxes, ha='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Side-by-side Metric Bar Chart
# ---------------------------------------------------------------
metrics     = ['Precision', 'Recall', 'F1-Score']
lr_scores   = [lr_precision,  lr_recall,  lr_f1]
svm_scores  = [svm_precision, svm_recall, svm_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, lr_scores,  width, label='Logistic Regression', color='#4C72B0', edgecolor='black')
bars2 = ax.bar(x + width/2, svm_scores, width, label='SVM (RBF)',           color='#DD8452', edgecolor='black')

# Add value labels on bars
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score (Fraud Class)')
ax.set_title('Model Comparison: Fraud Detection Metrics (Accuracy Excluded)', fontweight='bold')
ax.legend()
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='0.8 threshold')
plt.tight_layout()
plt.show()

print(f"\nFinal Comparison:")
print(f"{'Metric':<12} {'Logistic Reg':>14} {'SVM (RBF)':>12}")
print("-" * 40)
for m, l, s in zip(metrics, lr_scores, svm_scores):
    winner = '<-- better' if s > l else ''
    print(f"{m:<12} {l:>14.4f} {s:>12.4f}  {winner}")

---
## Step 9: Critical Analysis

### Why Did SVM (RBF) Outperform Logistic Regression?

**Logistic Regression** models the decision boundary as a linear hyperplane:
$$P(\text{Fraud} | x) = \sigma(w^T x + b)$$
This assumes that fraudulent and legitimate transactions are **linearly separable** in feature space.
In reality, the V1–V28 PCA-transformed features from real transaction patterns create **curved, non-linear
boundaries** between classes. Logistic Regression can approximate this with regularization, but it is
fundamentally constrained to a flat hyperplane.

**SVM with RBF Kernel** uses:
$$K(x, x') = \exp(-\gamma \|x - x'\|^2)$$
This kernel implicitly projects the data into an **infinite-dimensional** feature space where the
fraudulent cluster can be separated from legitimate transactions by a hyperplane — even if no such
plane exists in the original 29-dimensional space. The RBF kernel effectively measures **local similarity**
— nearby points in the original space have high kernel values, while distant points approach zero.
This allows the model to draw **smooth, curved islands** around fraud clusters.

In the original feature space, fraud patterns likely appear as **concentrated clusters** scattered
within the larger legitimate transaction manifold. The RBF kernel's Gaussian similarity measure
is perfectly suited to isolate such local concentrations.

---

### Why Is Recall More Important Than Precision for Fraud Detection?

Consider the two types of errors a fraud model can make:

| Error Type | What Happens | Consequence |
|------------|-------------|-------------|
| **False Positive** (Low Precision) | Legitimate transaction flagged as fraud | Customer is notified; minor inconvenience; transaction verified and cleared |
| **False Negative** (Low Recall) | Fraud transaction classified as legitimate | Bank loses money; customer's account compromised; trust destroyed |

For a bank:
- A **False Negative** means real fraud slips through — **direct financial loss** and **liability**.
- A **False Positive** triggers a fraud alert — the customer confirms the transaction. This is annoying
  but **recoverable**.

Therefore, **maximizing Recall** (catching as many actual fraud cases as possible) is the primary
objective, even at the cost of some Precision (accepting more false alarms).

The **F1-Score** provides a balanced view when both matter — but in practice, banks often optimize
a **weighted F-beta score** that places 2× or more weight on Recall:
$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \times \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$
where $\beta > 1$ prioritizes Recall over Precision.

In [ ]:
# ---------------------------------------------------------------
# Final Pipeline Summary
# ---------------------------------------------------------------
print("=" * 60)
print("CAPSTONE: CREDIT CARD FRAUD DETECTION — FINAL SUMMARY")
print("=" * 60)
print(f"Dataset       : {len(df):,} transactions | {y.sum():,} fraud ({y.mean()*100:.3f}%)")
print(f"SMOTE         : Fraud training samples {Counter(y_train)[1]} → {Counter(y_train_smote)[1]}")
print(f"Scaler        : StandardScaler (fitted on train only)")
print()
print(f"{'Model':<25} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 55)
print(f"{'Logistic Regression':<25} {lr_precision:>10.4f} {lr_recall:>8.4f} {lr_f1:>8.4f}")
print(f"{'SVM (RBF Kernel)':<25} {svm_precision:>10.4f} {svm_recall:>8.4f} {svm_f1:>8.4f}")
print()
print("Evaluation Metric: Precision, Recall, F1-Score (Accuracy excluded — Accuracy Paradox)")
print("Primary Metric   : Recall (missing fraud is more costly than a false alarm)")
print("=" * 60)